In [2]:
import numpy as np

In [3]:
# Reuse your Month 1 FIR Kaiser coefficients
# (regenerate if needed using your fir_bandpass_kaiser function's internals)
from scipy.signal import kaiserord, firwin

fs = 360
nyq = fs / 2
numtaps, beta = kaiserord(60, 1.5/nyq)
if numtaps % 2 == 0:
    numtaps += 1
    

fir_coeffs = firwin(numtaps, [0.5/nyq, 40/nyq], pass_zero=False, window=('kaiser', beta))

print(f"Number of taps: {len(fir_coeffs)}")
print(f"Coefficient range: {fir_coeffs.min():.6f} to {fir_coeffs.max():.6f}")

# ── Convert to Q1.15 fixed-point ─────────────────────────────────────────
def float_to_q15(x):
    scaled = np.round(x * 32768)
    return np.clip(scaled, -32768, 32767).astype(np.int16)

def fir_bandpass_kaiser(signal,lowcut,highcut,fs,ripple_db=60,width=1.5):
    nyq=fs/2
    
    # Kaiser formula: compute minimum taps and window shape (beta)
    numtaps,beta = kaiserord(ripple_db,width/nyq)
    if numtaps % 2==0:
        numtaps+=1         # FIR linear phase requires odd tap count
    
    if len(signal)<3*numtaps:
        print(f"Signal too short: {len(signal)} sample < 3 * {numtaps} taps")
        return None,numtaps
    
    print(f"kaiser formula -> {numtaps} taps | beta = {beta:.3f}")
    print(f"Filter memory -> {numtaps/fs :.3f} seconds")
    print(f"Signal length -> {len(signal)/fs:.1f} seconds")
    
    # Design FIR bandpass using Kaiser window
    taps=firwin(numtaps,[lowcut/nyq,highcut/nyq],
                pass_zero=False,window=("kaiser",beta))
    return filtfilt(taps,1.0,signal),numtaps

fir_coeffs_q15 = float_to_q15(fir_coeffs)

print(f"\nFirst 5 float coefficients : {fir_coeffs[:5]}")
print(f"First 5 Q1.15 coefficients: {fir_coeffs_q15[:5]}")

# Verify round-trip accuracy
fir_coeffs_reconstructed = fir_coeffs_q15 / 32768.0
max_error = np.max(np.abs(fir_coeffs - fir_coeffs_reconstructed))
print(f"\nMax quantization error: {max_error:.8f}")

Number of taps: 873
Coefficient range: -0.048700 to 0.219459

First 5 float coefficients : [1.42556742e-05 2.28164892e-05 2.54938755e-05 2.06392016e-05
 1.01669812e-05]
First 5 Q1.15 coefficients: [0 1 1 1 0]

Max quantization error: 0.00001521


In [4]:
center_idx = len(fir_coeffs) // 2
center_range = slice(center_idx - 5, center_idx + 5)

print(f"Center index: {center_idx}")
print(f"Center float coefficients:\n{fir_coeffs[center_range]}")
print(f"\nCenter Q1.15 coefficients:\n{fir_coeffs_q15[center_range]}")

# Reconstructed values at center
center_reconstructed = fir_coeffs_q15[center_range] / 32768.0
center_error = np.abs(fir_coeffs[center_range] - center_reconstructed)
print(f"\nCenter quantization error:\n{center_error}")
print(f"\nMax error at center: {center_error.max():.8f}")

Center index: 436
Center float coefficients:
[-0.0245439   0.02443621  0.08910569  0.15396113  0.20183836  0.21945877
  0.20183836  0.15396113  0.08910569  0.02443621]

Center Q1.15 coefficients:
[-804  801 2920 5045 6614 7191 6614 5045 2920  801]

Center quantization error:
[7.77098530e-06 8.37196092e-06 5.63556960e-06 5.11718987e-08
 4.90263793e-06 6.86153241e-06 4.90263793e-06 5.11718987e-08
 5.63556960e-06 8.37196092e-06]

Max error at center: 0.00000837
